## Training Speech AI with Mozilla Data Collective

This tutorial walks you through downloading, loading and finetuning Whisper on an MDC dataset end to end. In this notebook we will use as an example use-case the [Khmer](https://en.wikipedia.org/wiki/Khmer_language) language (an official and national language in Cambodia)  using the [Khmer ASR Cultural Dataset](https://datacollective.mozillafoundation.org/datasets/cmkcy8in2004umo0775mye43g) steward by [Digital Divide Data](https://www.digitaldividedata.com/)

Specifically, the flow of this tutorial focuses on:

1. Ensuring a GPU set up is available
2. Setting up and logging in to Mozilla Data Collective
3. Downloading and loading the dataset as a pandas DataFrame - with a single function call!
4. Generating an automated Exploratory Data Analysis (EDA) report to understand the dataset and inform our finetuning configuration
5. Configuring Whisper's fine-tuning hyper-parameters
6. Launching a fine-tuning job!

Note that steps 1-4 are only required the first time you set up your environment and download the dataset. Once you have the dataset downloaded and ready to use, you can skip directly to steps 5: to create a new set of hyper-parameters and 6: to start a new finetuning job!


[![Try Finetuning on Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mozilla-Data-Collective/speech-to-text-finetune/blob/mdc-demo/demo/khmer.ipynb) Run this notebook in Google Collab for limited, free GPU access!


### 1. (Optional) Google Collab setup and GPU check

If you are running this notebook in Google Collab you'll need to enable GPUs for the notebook: Navigate to Edit→Notebook Settings Select T4 GPU from the Hardware Accelerator section Click Save and accept. Next, we'll confirm that we can connect to the GPU:

In [ ]:
import torch

if not torch.cuda.is_available():
    print("GPU NOT available!")
else:
    print("GPU is available!")

_**Do not run the following command if you are not in a Google Collab environment**_

Next we will need to install the required dependencies for this notebook for this runtime environment. Run the follow command:

In [ ]:
!git clone -b mdc-demo --quiet https://github.com/Mozilla-Data-Collective/speech-to-text-finetune.git
%cd speech-to-text-finetune

!python -m pip install -U pip uv
!uv pip install --system -e . --group demo

### 2. Setup and login to Mozilla Data Collective

***(Required)*** In order to download any MDC dataset you will need to first create an account at Mozilla Data Collective and then get an API key.

1. Create a Mozilla Data Collective [account](https://datacollective.mozillafoundation.org/)
2. Get your API key by following the instructions [here](https://datacollective.mozillafoundation.org/api-reference)
3. Set your API key as an environment variable in your .env file or export it in your terminal. If you are running this notebook in Google Collab, you can set it for the session by running the cell below and entering your API key when prompted.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()  # this will load environment variables from your .env file if you have one

if os.getenv(
    "MDC_API_KEY"
):  # if API key is not set you can set it through the interactive jupyter terminal
    print("MDC API key found and loaded successfully!")
else:
    import getpass

    os.environ["MDC_API_KEY"] = getpass.getpass("Enter your MDC API key: ")

MDC API key found and loaded successfully!


***(Optional)*** If you want to track training and evaluation metrics of the finetuning and save your final model to use it and share it with others later, you will need a Hugging Face (HF) account.
1. Create a HF [account](https://huggingface.co/join)
2. Set up [personal access token](huggingface.co/settings/tokens)
3. Login to hugging face in this notebook by running the command below and using your token


In [ ]:
!huggingface-cli login

### 3. Download and load the Khmer dataset as a DataFrame

In [3]:
from datacollective import load_dataset

dataframe = load_dataset(
    "khmer-asr-cultural-dataset-4e33cd05",
    download_directory="local_data",
    enable_logging=True,
)

print("MDC dataframe preview:", dataframe.head())

2026-04-08 10:27:14,112 [INFO] [session=20260408T072713Z-pid31943-0e214920] datacollective.datasets: Loading dataset `khmer-asr-cultural-dataset-4e33cd05`
2026-04-08 10:27:15,130 [INFO] [session=20260408T072713Z-pid31943-0e214920] datacollective.download: File already exists. Skipping download: `local_data/khmer-asr-cultural-dataset-4e33cd05.tar.gz`
2026-04-08 10:27:15,134 [INFO] [session=20260408T072713Z-pid31943-0e214920] datacollective.archive_utils: Extracted directory already exists. Skipping extraction: `local_data/khmer-asr-cultural-dataset-4e33cd05`
2026-04-08 10:27:15,148 [INFO] [session=20260408T072713Z-pid31943-0e214920] datacollective.schema_loaders.cache_schema: Archive checksum matches cached schema – skipping schema download.
2026-04-08 10:27:15,152 [INFO] [session=20260408T072713Z-pid31943-0e214920] datacollective.schema_loaders.registry: Loading dataset 'cmkcy8in2004umo0775mye43g' with ASRLoader


MDC dataframe preview:                                           audio_path  \
0  /home/kostis/Projects/MDC/mdc-stt-finetune/dem...   
1  /home/kostis/Projects/MDC/mdc-stt-finetune/dem...   
2  /home/kostis/Projects/MDC/mdc-stt-finetune/dem...   
3  /home/kostis/Projects/MDC/mdc-stt-finetune/dem...   
4  /home/kostis/Projects/MDC/mdc-stt-finetune/dem...   

                                       transcription    topic  \
0  មុខម្ហូបតាមដងផ្លូវ គឺជាមុខម្ហូបមួយមានភាពសម្បូរ...  Recipes   
1  នៅក្នុងប្រទេសកម្ពុជា មុខម្ហូបតាមដងផ្លូវមានប្រជ...  Recipes   
2  ម្យ៉ាងវិញទៀត អ្នកអាចរកទិញមុខម្ហូបទាំងនោះបានយ៉ា...  Recipes   
3  ម្យ៉ាងទៀតម្ហូបទាំងនោះ គេតែងតែបរិភោគភាគច្រើននៅព...  Recipes   
4  ដូចជានំបញ្ចុក គេពេញនិយមបរិភោគសម្រាប់អាហារពេលព្...  Recipes   

             subtopic   speaker_id paragraph_id  
0  Street food dishes  f-adt1-0001            1  
1  Street food dishes  f-adt1-0001            1  
2  Street food dishes  f-adt1-0001            1  
3  Street food dishes  f-adt1-0001            1  

### 4. Exploratory Data Analysis (EDA) with ydata-profiling

In [4]:
from pathlib import Path

if Path("khmer_dataset_report.html").exists():
    print("EDA report already exists, skipping generation...")
else:
    from ydata_profiling import ProfileReport

    # The audio_path values are causing an issue with the library rendering the report so we remove them
    dataframe_no_audio_path = dataframe.drop(columns=["audio_path"])
    profile = ProfileReport(dataframe_no_audio_path, title="Profiling Report")
    profile.to_file("khmer_dataset_report.html")
    del dataframe_no_audio_path  # we delete the intermediate dataframe to save memory
del dataframe

EDA report already exists, skipping generation...


### 5. Configure hyper-parameters for finetuning

In [5]:
# @title Finetuning configuration and hyperparameter setting
import yaml


def save_to_yaml(filename="config.yaml"):
    with open(filename, "w") as file:
        yaml.dump(cfg, file)


model_id = "openai/whisper-tiny"  # @param ["openai/whisper-tiny", "openai/whisper-small", "openai/whisper-medium","openai/whisper-large-v3"]
dataset_id = "khmer-asr-cultural-dataset-4e33cd05"  # @param {type: "string"}
language = "Khmer"  # @param {type: "string"}
repo_name = "default"  # @param {type: "string"}
push_to_hub = False  # @param {type: 'boolean'}
n_train_samples = 25  # @param {type: "int"}
n_test_samples = 10  # @param {type: "int"}
download_directory = "local_data"  # @param {type: "string"}
hub_private_repo = True  # @param {type: 'boolean'}
max_steps = 25  # @param {type: "slider", min: 1, max: 3000, step: 10}
per_device_train_batch_size = 8  # @param {type: "slider", min: 1, max: 300}
gradient_accumulation_steps = 1  # @param {type: "slider", min: 1, max: 10}
warmup_steps = 5  # @param {type: "slider", min: 0, max: 500}
gradient_checkpointing = True  # @param {type: 'boolean'}
fp16 = True  # @param {type: 'boolean'}
per_device_eval_batch_size = 4  # @param {type: "slider", min: 1, max: 200}
save_steps = 5  # @param {type: "slider", min: 1, max: 500}
logging_steps = 5  # @param {type: "slider", min: 1, max: 500}
load_best_model_at_end = True  # @param {type: 'boolean'}

cfg = {
    "model_id": model_id,
    "dataset_id": dataset_id,
    "language": language,
    "repo_name": repo_name,
    "n_train_samples": n_train_samples,
    "n_test_samples": n_test_samples,
    "download_directory": download_directory,
    "training_hp": {
        "push_to_hub": push_to_hub,
        "hub_private_repo": hub_private_repo,
        "max_steps": max_steps,
        "per_device_train_batch_size": per_device_train_batch_size,
        "gradient_accumulation_steps": gradient_accumulation_steps,
        "learning_rate": 1e-5,
        "warmup_steps": warmup_steps,
        "gradient_checkpointing": gradient_checkpointing,
        "fp16": fp16,
        "eval_strategy": "steps",
        "per_device_eval_batch_size": per_device_eval_batch_size,
        "predict_with_generate": True,
        "generation_max_length": 225,
        "save_steps": save_steps,
        "logging_steps": logging_steps,
        "load_best_model_at_end": load_best_model_at_end,
        "save_total_limit": 1,
        "metric_for_best_model": "wer",
        "greater_is_better": False,
    },
}

save_to_yaml()

### 6. Start finetuning job

Note that this might take a while, anything from 10min to 10hours depending on your model choice and hyper-parameter configuration

In [6]:
from speech_to_text_finetune.finetune_whisper import run_finetuning

run_finetuning(config_path="config.yaml")

/home/kostis/Projects/MDC/mdc-stt-finetune/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-08 10:27:56.027 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:65 - Finetuning starts soon, results saved locally at ./artifacts/whisper-tiny-km
2026-04-08 10:27:56.155 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:76 - Loading openai/whisper-tiny on NVIDIA GeForce RTX 2060 SUPER.
2026-04-08 10:27:57.028 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:82 - Configuring openai/whisper-tiny for Khmer
2026-04-08 10:27:59.044 | INFO     | speech_to_text_finetune.data_process:try_find_processed_version:50 - Found processed dataset version at artifacts/khmer-asr-cultural-dataset-4e33cd05/processed_version of MDC dataset khmer-asr-cultural-dat

2026-04-08 10:28:05.972 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:166 - Baseline evaluation complete. Results:
	 {'eval_loss': 3.050732374191284, 'eval_model_preparation_time': 0.0019, 'eval_wer_ortho': 353.48837209302326, 'eval_wer': 117.3076923076923, 'eval_cer_ortho': 119.45031712473573, 'eval_cer': 105.62913907284768, 'eval_runtime': 4.0341, 'eval_samples_per_second': 2.479, 'eval_steps_per_second': 0.744}
2026-04-08 10:28:05.976 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:168 - Start finetuning job on 25 audio samples. Monitor training metrics in real time in a local tensorboard server by running in a new terminal: tensorboard --logdir ./artifacts/whisper-tiny-km/runs


Step,Training Loss,Validation Loss,Model Preparation Time,Wer Ortho,Wer,Cer Ortho,Cer
5,2.956300,3.050732,0.001900,353.488372,117.307692,119.450317,105.629139
10,2.639000,2.407246,0.001900,348.837209,124.725275,138.900634,121.523179
15,2.281200,2.263956,0.001900,502.325581,107.692308,150.845666,136.644592
20,2.275000,2.199410,0.001900,427.906977,100.000000,158.562368,148.454746
25,2.229400,2.179554,0.001900,420.930233,100.000000,159.302326,149.558499


/home/kostis/Projects/MDC/mdc-stt-finetune/.venv/lib/python3.13/site-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50358, 50359, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['proj_out.w

2026-04-08 10:28:53.091 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:181 - Evaluation complete. Results:
	 {'eval_loss': 2.1994102001190186, 'eval_model_preparation_time': 0.0019, 'eval_wer_ortho': 427.906976744186, 'eval_wer': 100.0, 'eval_cer_ortho': 158.56236786469344, 'eval_cer': 148.45474613686534, 'eval_runtime': 3.7804, 'eval_samples_per_second': 2.645, 'eval_steps_per_second': 0.794, 'epoch': 6.25}
2026-04-08 10:28:53.094 | INFO     | speech_to_text_finetune.finetune_whisper:run_finetuning:202 - Find your final, best performing model at ./artifacts/whisper-tiny-km


({'eval_loss': 3.050732374191284,
  'eval_model_preparation_time': 0.0019,
  'eval_wer_ortho': 353.48837209302326,
  'eval_wer': 117.3076923076923,
  'eval_cer_ortho': 119.45031712473573,
  'eval_cer': 105.62913907284768,
  'eval_runtime': 4.0341,
  'eval_samples_per_second': 2.479,
  'eval_steps_per_second': 0.744},
 {'eval_loss': 2.1994102001190186,
  'eval_model_preparation_time': 0.0019,
  'eval_wer_ortho': 427.906976744186,
  'eval_wer': 100.0,
  'eval_cer_ortho': 158.56236786469344,
  'eval_cer': 148.45474613686534,
  'eval_runtime': 3.7804,
  'eval_samples_per_second': 2.645,
  'eval_steps_per_second': 0.794,
  'epoch': 6.25})